## LIGHTGBM GOSS

In [ ]:
X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train) #Model without feature selection
#X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train_wrapper) #Model with feature selection

D = [X1, X2, X3, X4, X5]

In [ ]:
# Define the columns for the results DataFrame
columns = ['Model', 'Top_rate', 'Other_rate', 'Learning_rate', 'Num_leaves',
           'Accuracy',
           'Recall',
           'Specificity',
           'Precision',
           'F1']
df_results = pd.DataFrame(columns=columns)

# Define parameters for Grid Search
top_rate = [0.2, 0.4, 0.6]
other_rate = [0.05, 0.1, 0.3]
learning_rates = [0.025, 0.05, 0.1, 0.2]
num_leaves = [10, 30, 50]
model_name = 'LightGBM GOSS'

for i in top_rate:
    for d in learning_rates:
        for k in other_rate:
            for l in num_leaves:
              df_results_fold = pd.DataFrame(columns=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1'])
                for j in range(5):
                    # Prepare test and train sets for this fold
                    d_test = pd.concat([D[j]])
                    y_test = d_test['label_binary']
                    X_test = d_test.drop(columns=['label_binary', 'n_image', 'label_multi'])
                    d_train = pd.concat([D[k] for k in range(5) if k != j], ignore_index=True)
                    y_train = d_train['label_binary']
                    X_train = d_train.drop(columns=['label_binary', 'n_image', 'label_multi'])
                    label_mapping = {'no corrosion': 0, 'corrosion': 1}
                    y_train = y_train.map(label_mapping)
                    y_test = y_test.map(label_mapping)

                    # Initialize and train the LightGBM model
                    model = lgbm(
                        objective='binary',  # Binary classification
                        boosting_type='goss',  # Boosting type
                        top_rate=i,             # Top rate
                        seed=42,                       # For reproducibility
                        learning_rate=d,
                        other_rate=k,
                        num_leaves=l,
                        verbose=-1
                    )
                    model.fit(X_train, y_train)

                    # Measure execution time
                    t0 = time.time()
                    y_pred = model.predict(X_test)
                    print(f'Model with {i} top rate, {d} learning rate, and {l} leaves:')

                    df_results = pd.DataFrame(columns=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1'])
                    results=evaluate_model(y_pred, y_test, model=model_name, labels=(1,0))
                    df_results_fold = pd.concat([df_results_fold, results], ignore_index=True)
                    print('\n')

                # Calculate mean metrics across folds
                recall_mean, specificity_mean, precision_mean, f1_mean, accuracy_mean = means_results(df_results)


                # Append results to DataFrame
                result_i = {'Model': model_name, 'Accuracy': accuracy_mean, 'Top_rate': i, 'Learning_rate': d, 'Other_rate': k, 'Num_leaves': l,
                            'Recall': recall_mean, 'Specificity': specificity_mean,
                            'Precision': precision_mean, 'F1': f1_mean}
                df_results = pd.concat([df_results, pd.DataFrame([result_i])], ignore_index=True)


In [ ]:
df_results.sort_values(by='Recall', ascending=False)

Chosen model

In [ ]:
# Model parameters
top_rate = 0.2
learning_rate = 0.05
other_rate = 0.05
num_leaves = 10

# Prepare training and testing data
X_train = data_train.drop(columns=['label_binary', 'label_multi', 'n_image'])
y_train = data_train['label_binary']
label_mapping = {'no corrosion': 0, 'corrosion': 1}
y_train = y_train.map(label_mapping)

X_test = data_test.drop(columns=['label_binary', 'label_multi', 'n_image'])
y_test = data_test['label_binary']
y_test = y_test.map(label_mapping)

# Initialize and train the LightGBM model
model = lgbm(
    objective='binary',  # Binary classification
    boosting_type='goss',  # Boosting type
    top_rate=top_rate,             # Top rate
    seed=42,                       # For reproducibility
    learning_rate=learning_rate,
    other_rate=other_rate,
    num_leaves=num_leaves,
    verbose=-1
)
model.fit(X_train, y_train)

# Measure execution time
t0 = time.time()
y_pred = model.predict(X_test)
t1 = time.time()

# Print model details and metrics
print(f'Model with {num_leaves} leaves, {learning_rate} learning rate, {top_rate} top rate, and {other_rate} other rate:')
labels = (1, 0)
cm = confusion_matrix(y_test, y_pred, labels=labels)
print(f'Confusion matrix: \n {cm}')
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
recall = recall_score(y_test, y_pred, average='binary')
recall = round(recall, 3)
print(f'Recall: {recall}')
specificity = recall_score(y_test, y_pred, average='binary', pos_label=0)
specificity = round(specificity, 3)
print(f'Specificity: {specificity}')
precision = precision_score(y_test, y_pred, average='binary')
precision = round(precision, 3)
print(f'Precision: {precision}')
f1 = f1_score(y_test, y_pred, average='binary')
f1 = round(f1, 3)
print(f'F1: {f1}')
time_taken = t1 - t0
time_taken = round(time_taken, 3)
print(f'Execution time: {time_taken} seconds')
print('\n')

# Create a DataFrame to store results
columns = ['Model', 'Learning_rate', 'Num_leaves', 'Top_rate', 'Other_rate',
           'Accuracy',
           'Recall',
           'Specificity',
           'Precision',
           'F1',
           'Time']
df_results = pd.DataFrame(columns=columns)

# Append results to DataFrame
result_lgbm = {'Model': 'LightGBM', 'Accuracy': accuracy, 'Top_rate': top_rate, 'Other_rate': other_rate, 'Learning_rate': learning_rate, 'Num_leaves': num_leaves,
               'Recall': recall, 'Specificity': specificity, 'Precision': precision, 'F1': f1,
               'Time': time_taken}
df_results = pd.concat([df_results, pd.DataFrame([result_lgbm])], ignore_index=True)
print(df_results)
